In [4]:
!pip install symspellpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 14.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [symspellpy]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [5]:
from __future__ import annotations

from random import Random
from unicodedata import normalize

from symspellpy import SymSpell, Verbosity

RNG = Random(123)

HEALTH_BASE = [
    'hastane', 'doktor', 'hekim', 'hemşire', 'ilaç', 'reçete', 'muayene',
    'randevu', 'tedavi', 'ameliyat', 'enfeksiyon', 'aşı', 'ateş', 'öksürük',
    'boğaz', 'solunum', 'kalp', 'damar', 'beyin', 'sinir', 'cilt', 'kulak',
    'burun', 'göz', 'diş', 'diyabet', 'tansiyon', 'obezite', 'kanser',
    'nefroloji', 'kardiyoloji', 'nöroloji', 'ortopedi', 'psikiyatri', 'acil',
    'yoğunbakım', 'laboratuvar', 'rapor', 'tahlil', 'semptom', 'teşhis'
]

HEALTH_COMPOUNDS = [
    'acilservis', 'ağrıkesici', 'kanbasıncı', 'kalpkrizi', 'beyincerrahisi',
    'gözmuayenesi', 'kulakburunboğaz', 'aşılama', 'ilaçtakibi', 'randevusistemi',
    'hastakaydı', 'tahlilsonucu', 'tedaviplanı', 'solunumyolu', 'doktorraporu',
    'hemşirelik', 'ameliyatöncesi', 'ameliyatsonrası', 'kanşekeri', 'nabızölçer'
]

HEALTH_SPLIT_PARTS = [
    'servis', 'ağrı', 'kesici', 'kan', 'basıncı', 'kalp', 'krizi', 'beyin', 'cerrahisi',
    'göz', 'muayenesi', 'kulak', 'burun', 'boğaz', 'aşı', 'lama', 'ilaç', 'takibi',
    'randevu', 'sistemi', 'hasta', 'kaydı', 'tahlil', 'sonucu', 'tedavi', 'planı',
    'solunum', 'yolu', 'doktor', 'raporu', 'ameliyat', 'öncesi', 'sonrası', 'kan',
    'şekeri', 'nabız', 'ölçer', 'bakım', 'aşılama'
]

DICTIONARY_WORDS = sorted(set(HEALTH_BASE + HEALTH_COMPOUNDS + HEALTH_SPLIT_PARTS))

max_edit_distance = 2
prefix_length = 7
sym_spell = SymSpell(max_dictionary_edit_distance=max_edit_distance, prefix_length=prefix_length)
split_spell = SymSpell(max_dictionary_edit_distance=max_edit_distance, prefix_length=prefix_length)


for term in DICTIONARY_WORDS:
    if term in HEALTH_BASE:
        freq = 20
    elif term in HEALTH_COMPOUNDS:
        freq = 10
    else:
        freq = 5
    sym_spell.create_dictionary_entry(term, freq)
    if term in HEALTH_BASE or term in HEALTH_SPLIT_PARTS:
        split_spell.create_dictionary_entry(term, freq)


def remove_diacritics(text: str) -> str:
    return normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')


def typo_transform(word: str, forced_type: str | None = None) -> tuple[str, str]:
    typo_type = forced_type or RNG.choice(['diacritic', 'transpose', 'delete', 'replace', 'double'])

    if typo_type == 'diacritic':
        return remove_diacritics(word), typo_type

    if typo_type == 'transpose' and len(word) > 2:
        i = RNG.randrange(len(word) - 1)
        letters = list(word)
        letters[i], letters[i + 1] = letters[i + 1], letters[i]
        return ''.join(letters), typo_type

    if typo_type == 'delete' and len(word) > 2:
        i = RNG.randrange(len(word))
        return word[:i] + word[i + 1:], typo_type

    if typo_type == 'replace' and len(word) > 0:
        i = RNG.randrange(len(word))
        replacement = 'a' if word[i] != 'a' else 'e'
        return word[:i] + replacement + word[i + 1:], typo_type

    if typo_type == 'double' and len(word) > 0:
        i = RNG.randrange(len(word))
        return word[:i] + word[i] + word[i:], typo_type

    return word, 'none'


def correct_single_word(typo: str) -> tuple[str | None, int]:
    suggestions = sym_spell.lookup(typo, Verbosity.CLOSEST, max_edit_distance)
    if not suggestions:
        return None, -1
    return suggestions[0].term, suggestions[0].distance


def correct_split_phrase(typo_phrase: str) -> tuple[str | None, int]:
    suggestions = split_spell.lookup_compound(typo_phrase, max_edit_distance)
    if not suggestions:
        return None, -1
    return suggestions[0].term, suggestions[0].distance


SINGLE_WORD_EXAMPLES = [
    'hastane', 'doktor', 'hemşire', 'ilaç', 'reçete', 'muayene', 'tedavi',
    'ameliyat', 'enfeksiyon', 'aşı', 'ateş', 'öksürük', 'kalp', 'beyin'
]

COMPOUND_EXAMPLES = [
    ('acilservis', 'acilserviz'),
    ('ağrıkesici', 'agrikesici'),
    ('kanbasıncı', 'kanbasınc'),
    ('kalpkrizi', 'kalpkrizii'),
    ('beyincerrahisi', 'beyincerrahis'),
    ('gözmuayenesi', 'gozmuayenesi'),
    ('kulakburunboğaz', 'kulakburunbogaz'),
    ('ilaçtakibi', 'ilactakibi'),
    ('randevusistemi', 'randevusistem'),
    ('hastakaydı', 'hastakayyı'),
    ('tahlilsonucu', 'tahlilsonuu'),
    ('doktorraporu', 'doktorraprpu'),
    ('solunumyolu', 'solunumyoluu'),
    ('kanşekeri', 'kansekeri'),
    ('nabızölçer', 'nabızölçre'),
]

SPLIT_PHRASE_EXAMPLES = [
    ('acil servis', 'acilservis'),
    ('ağrı kesici', 'ağrikesici'),
    ('kan basıncı', 'kanbasinci'),
    ('kalp krizi', 'kalpkrizi'),
    ('beyin cerrahisi', 'beyincerrahisi'),
    ('göz muayenesi', 'gozmuayenesi'),
    ('ilaç takibi', 'ilactakibi'),
    ('randevu sistemi', 'randevusistemi'),
    ('hasta kaydı', 'hastakaydi'),
    ('tahlil sonucu', 'tahlilsonucu'),
    ('doktor raporu', 'doktorraporu'),
    ('solunum yolu', 'solunumyolu'),
    ('kan şekeri', 'kansekeri'),
    ('nabız ölçer', 'nabızolcer'),
    ('ameliyat öncesi', 'ameliyatoncesi'),
    ('ameliyat sonrası', 'ameliyatsonrasi'),
    ('aşılama planı', 'aşilamaplani'),
    ('laboratuvar raporu', 'laboratuvarraporu'),
    ('psikiyatri randevu', 'psikiyatrirandevu'),
    ('ortopedi raporu', 'ortopediraporu'),
    ('nefroloji muayenesi', 'nefrolojimuayenesi'),
    ('kardiyoloji randevu', 'kardiyolojirandevu'),
    ('teşhis raporu', 'teshisraporu'),
]

print(f'Synthetic health dictionary size: {len(DICTIONARY_WORDS)}')
print(f'Single-word examples: {len(SINGLE_WORD_EXAMPLES)}')
print(f'Compound-word examples: {len(COMPOUND_EXAMPLES)}')
print(f'Split-phrase examples: {len(SPLIT_PHRASE_EXAMPLES)}')
print('-' * 100)

print('TEKIL KELIME DUZELTME ORNEKLERI')
print('-' * 100)
for gold in SINGLE_WORD_EXAMPLES:
    typo, typo_type = typo_transform(gold)
    suggestion, distance = correct_single_word(typo)
    status = 'OK' if suggestion == gold else 'FAIL'
    print(f'{gold:20s} | typo={typo:20s} | sug={str(suggestion):20s} | d={distance:2d} | {typo_type:10s} | {status}')

print('-' * 100)
print('HATALI BIRLESIK KELIME ORNEKLERI')
print('-' * 100)
for gold, typo in COMPOUND_EXAMPLES:
    suggestion, distance = correct_single_word(typo)
    status = 'OK' if suggestion == gold else 'FAIL'
    print(f'{gold:20s} | typo={typo:20s} | sug={str(suggestion):20s} | d={distance:2d} | {status}')

print('-' * 100)
print('NORMALDE AYRI YAZILMASI GEREKEN HATALI ORNEKLER')
print('-' * 100)
for gold_phrase, typo_phrase in SPLIT_PHRASE_EXAMPLES:
    suggestion, distance = correct_split_phrase(typo_phrase)
    status = 'OK' if suggestion == gold_phrase else 'FAIL'
    print(f'{gold_phrase:20s} | typo={typo_phrase:20s} | sug={str(suggestion):20s} | d={distance:2d} | {status}')

print('-' * 100)


Synthetic health dictionary size: 84
Single-word examples: 14
Compound-word examples: 15
Split-phrase examples: 23
----------------------------------------------------------------------------------------------------
TEKIL KELIME DUZELTME ORNEKLERI
----------------------------------------------------------------------------------------------------
hastane              | typo=hastane              | sug=hastane              | d= 0 | diacritic  | OK
doktor               | typo=oktor                | sug=doktor               | d= 1 | delete     | OK
hemşire              | typo=heaşire              | sug=hemşire              | d= 1 | replace    | OK
ilaç                 | typo=ilac                 | sug=ilaç                 | d= 1 | diacritic  | OK
reçete               | typo=recete               | sug=reçete               | d= 1 | diacritic  | OK
muayene              | typo=muayane              | sug=muayene              | d= 1 | replace    | OK
tedavi               | typo=teddavi          